In [2]:
# Fase 2 - PySpark, Puerto Coquimbo sitio 3
# Hito 1.1: Parametros, rutas y valores esperados del dominio
import glob
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType,
)

RUTA_PROYECTO = "/opt/workspace"

# subcarpeta de la fase 1 que se procesa (w01, w02, w04, w08 o w16)
CONFIG_FASE1 = "w16"

RUTA_RAW = f"{RUTA_PROYECTO}/data/raw/{CONFIG_FASE1}"
PATRON_RAW = f"{RUTA_RAW}/*.jsonl"
RUTA_SALIDA = f"{RUTA_PROYECTO}/salida_fase2"   # en .gitignore, se usa en el hito 5

# valores esperados del dominio, salen de src/simulador.py (_inventario)
EQUIPOS_ESPERADOS = 69
PUNTOS_ESPERADOS = 185
METRICAS_ESPERADAS = {"vibration", "current", "weight", "temperature", "humidity"}
EVENTOS_ESPERADOS = 10_000_000   # src/main.py, TOTAL_EVENTOS

In [3]:
# Hito 1.2: Comprobacion previa de los .jsonl desde el driver
archivos = sorted(glob.glob(PATRON_RAW))
if not archivos:
    raise FileNotFoundError(
        f"No hay .jsonl en {RUTA_RAW}. Revisa CONFIG_FASE1 o genera los datos "
        "con la fase 1 (python src/main.py)."
    )

total_bytes = sum(os.path.getsize(a) for a in archivos)
print(f"Archivos : {len(archivos)}")
print(f"Volumen  : {total_bytes / 1024**3:.2f} GiB")
for a in archivos[:5]:
    print(f"  {os.path.basename(a):<16} {os.path.getsize(a) / 1024**2:8.1f} MiB")
if len(archivos) > 5:
    print(f"  ... y {len(archivos) - 5} mas")

Archivos : 16
Volumen  : 1.18 GiB
  part-000.jsonl       84.6 MiB
  part-001.jsonl       85.0 MiB
  part-002.jsonl       85.0 MiB
  part-003.jsonl       85.0 MiB
  part-004.jsonl       84.6 MiB
  ... y 11 mas


In [4]:
# Hito 1.3: Sesion Spark contra el cluster y espera de executors
spark = (
    SparkSession.builder
    .appName("fase2-puerto-coquimbo")
    .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "1g"))
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

# los executors tardan unos segundos en registrarse
esperados = int(os.environ.get("SPARK_WORKERS", "1"))
for _ in range(30):
    ejecutores = sc._jsc.sc().getExecutorMemoryStatus().size() - 1   # -1 = el driver
    if ejecutores >= esperados:
        break
    time.sleep(1)

print(f"Spark            : {spark.version}")
print(f"Master           : {sc.master}")
print(f"Executors vivos  : {ejecutores} (esperados {esperados})")
print(f"Paralelismo      : {sc.defaultParallelism}")
print(f"Zona horaria     : {spark.conf.get('spark.sql.session.timeZone')}")
print(f"AQE              : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"maxPartitionBytes: {spark.conf.get('spark.sql.files.maxPartitionBytes')}")
if ejecutores == 0:
    raise RuntimeError("Sin executors: revisa SPARK_EXECUTOR_MEMORY <= SPARK_WORKER_MEMORY")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 02:02:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark            : 4.0.1
Master           : spark://spark-master:7077
Executors vivos  : 3 (esperados 3)
Paralelismo      : 6
Zona horaria     : UTC
AQE              : true
maxPartitionBytes: 134217728b


In [5]:
# Hito 1.4: Esquema explicito (timestamp como texto, _corrupt_record para el diagnostico)
ESQUEMA_EVENTO = StructType([
    StructField("timestamp",       StringType(),  True),
    StructField("sensor_id",       StringType(),  True),
    StructField("metric",          StringType(),  True),
    StructField("value",           DoubleType(),  True),
    StructField("unit",            StringType(),  True),
    StructField("worker_id",       IntegerType(), True),
    StructField("_corrupt_record", StringType(),  True),
])

In [6]:
# Hito 1.5: Lectura PERMISSIVE con columna de corruptos (perezosa)
df_raw = (
    spark.read
    .schema(ESQUEMA_EVENTO)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(PATRON_RAW)
)

df_raw.printSchema()
print(f"Particiones de lectura: {df_raw.rdd.getNumPartitions()}")

26/10/04 02:02:24 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /opt/workspace/data/raw/w16/*.jsonl.
java.io.FileNotFoundException: File /opt/workspace/data/raw/w16/*.jsonl does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql

root
 |-- timestamp: string (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- worker_id: integer (nullable = true)
 |-- _corrupt_record: string (nullable = true)

Particiones de lectura: 16


In [7]:
# Hito 1.6: Muestra de la ingesta
df_raw.show(5, truncate=False)

[Stage 0:>                                                          (0 + 1) / 1]

+------------------------+------------+-----------+------+----+---------+---------------+
|timestamp               |sensor_id   |metric     |value |unit|worker_id|_corrupt_record|
+------------------------+------------+-----------+------+----+---------+---------------+
|2026-01-01T00:00:00.000Z|FAJA_S3_C3  |vibration  |1.773 |mm/s|2        |NULL           |
|2026-01-01T00:00:00.000Z|FAJA_S3_C3  |current    |26.2  |A   |2        |NULL           |
|2026-01-01T00:00:00.000Z|REEFER_S3_10|temperature|-17.99|C   |2        |NULL           |
|2026-01-01T00:00:00.000Z|REEFER_S3_10|current    |7.47  |A   |2        |NULL           |
|2026-01-01T00:00:00.000Z|REEFER_S3_10|humidity   |76.6  |%   |2        |NULL           |
+------------------------+------------+-----------+------+----+---------+---------------+
only showing top 5 rows


In [8]:
# Hito 1.6: Cache y conteo total, primera accion sobre todo el dataset
df_raw = df_raw.cache()

t0 = time.perf_counter()
total = df_raw.count()
t_lectura = time.perf_counter() - t0

print(f"Registros leidos : {total:,}")
print(f"Esperados        : {EVENTOS_ESPERADOS:,}")
print(f"Tiempo (lectura + cache): {t_lectura:.1f} s")
if total != EVENTOS_ESPERADOS:
    print("AVISO: el conteo no coincide con TOTAL_EVENTOS de la fase 1")

[Stage 1:=====================================================>   (15 + 1) / 16]

Registros leidos : 10,000,000
Esperados        : 10,000,000
Tiempo (lectura + cache): 10.2 s


In [9]:
# Hito 2: Perfilado y diagnostico de calidad (solo mide, no corrige)
# paso del reloj simulado de la fase 1 (src/generador.py, DT_MS = 15_000)
DT_S = 15

# unidad que la fase 1 le asigna a cada metrica (src/simulador.py)
UNIDAD_POR_METRICA = {
    "vibration": "mm/s", "current": "A", "weight": "kg",
    "temperature": "C", "humidity": "%",
}

# formato exacto que escribe la fase 1; se valida como texto, el parseo es del hito 3
PATRON_TS = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}Z$"

diagnostico = {}   # chequeo -> (valor medido, esperado, ok)
tiempos_h2 = {}    # seccion -> segundos de pared


def registrar(chequeo, medido, esperado, ok=None):
    ok = (medido == esperado) if ok is None else ok
    diagnostico[chequeo] = (medido, esperado, ok)
    print(f"[{'OK' if ok else 'FALLA'}] {chequeo}: {medido} (esperado {esperado})")

In [10]:
# Hito 2.1: Estado del cache de df_raw
for info in sc._jsc.sc().getRDDStorageInfo():
    print(f"RDD          : {info.name()[:60]}")
    print(f"Particiones  : {info.numCachedPartitions()} de {info.numPartitions()} cacheadas")
    print(f"En memoria   : {info.memSize() / 1024**2:,.1f} MiB")
    print(f"En disco     : {info.diskSize() / 1024**2:,.1f} MiB")

RDD          : FileScan json [timestamp#0,sensor_id#1,metric#2,value#3,unit
Particiones  : 16 de 16 cacheadas
En memoria   : 199.6 MiB
En disco     : 0.0 MiB


In [11]:
# Hito 2.2: Validez por linea en un solo agg (corruptas, nulos, formato, rangos)
t0 = time.perf_counter()

columnas = ["timestamp", "sensor_id", "metric", "value", "unit", "worker_id"]
fuera_de_rango = (
    (F.col("metric").isin("current", "vibration", "weight") & (F.col("value") < 0))
    | ((F.col("metric") == "humidity") & ~F.col("value").between(0, 100))
)

fila = df_raw.agg(
    F.count(F.col("_corrupt_record")).alias("corruptas"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"nulos_{c}") for c in columnas],
    F.sum((~F.col("timestamp").rlike(PATRON_TS)).cast("int")).alias("ts_mal_formado"),
    F.sum(F.isnan("value").cast("int")).alias("value_nan"),
    F.sum(fuera_de_rango.cast("int")).alias("fuera_de_rango_fisico"),
).first().asDict()

tiempos_h2["2.2 validez por linea"] = time.perf_counter() - t0

registrar("lineas corruptas", fila["corruptas"], 0)
for c in columnas:
    registrar(f"nulos en {c}", fila[f"nulos_{c}"], 0)
registrar("timestamp mal formado", fila["ts_mal_formado"], 0)
registrar("value NaN", fila["value_nan"], 0)
registrar("value fisicamente imposible", fila["fuera_de_rango_fisico"], 0)
print(f"\nTiempo: {tiempos_h2['2.2 validez por linea']:.1f} s")

[Stage 5:=================================================>       (14 + 2) / 16]

[OK] lineas corruptas: 0 (esperado 0)
[OK] nulos en timestamp: 0 (esperado 0)
[OK] nulos en sensor_id: 0 (esperado 0)
[OK] nulos en metric: 0 (esperado 0)
[OK] nulos en value: 0 (esperado 0)
[OK] nulos en unit: 0 (esperado 0)
[OK] nulos en worker_id: 0 (esperado 0)
[OK] timestamp mal formado: 0 (esperado 0)
[OK] value NaN: 0 (esperado 0)
[OK] value fisicamente imposible: 0 (esperado 0)

Tiempo: 2.6 s


In [12]:
# Hito 2.2: Muestra de lineas corruptas, si las hay
if fila["corruptas"]:
    df_raw.filter(F.col("_corrupt_record").isNotNull()) \
          .select("_corrupt_record").show(5, truncate=False)

In [13]:
# Hito 2.3: Dominio: equipos, series, metricas y unidades
t0 = time.perf_counter()

dominio = df_raw.agg(
    F.countDistinct("sensor_id").alias("equipos"),
    F.countDistinct("sensor_id", "metric").alias("series"),
    F.collect_set("metric").alias("metricas"),
).first()

unidades = (
    df_raw.groupBy("metric")
    .agg(F.collect_set("unit").alias("unidades"), F.count("*").alias("lecturas"))
    .orderBy("metric")
    .collect()
)

tiempos_h2["2.3 dominio"] = time.perf_counter() - t0

registrar("equipos distintos", dominio["equipos"], EQUIPOS_ESPERADOS)
registrar("series (sensor_id, metric)", dominio["series"], PUNTOS_ESPERADOS)
registrar("metricas", sorted(dominio["metricas"]), sorted(METRICAS_ESPERADAS))

print()
for r in unidades:
    esperada = UNIDAD_POR_METRICA.get(r["metric"])
    registrar(f"unidad de {r['metric']}", sorted(r["unidades"]), [esperada])
    print(f"      lecturas: {r['lecturas']:,}")
print(f"\nTiempo: {tiempos_h2['2.3 dominio']:.1f} s")

[OK] equipos distintos: 69 (esperado 69)
[OK] series (sensor_id, metric): 185 (esperado 185)
[OK] metricas: ['current', 'humidity', 'temperature', 'vibration', 'weight'] (esperado ['current', 'humidity', 'temperature', 'vibration', 'weight'])

[OK] unidad de current: ['A'] (esperado ['A'])
      lecturas: 3,027,029
[OK] unidad de humidity: ['%'] (esperado ['%'])
      lecturas: 3,243,240
[OK] unidad de temperature: ['C'] (esperado ['C'])
      lecturas: 3,243,240
[OK] unidad de vibration: ['mm/s'] (esperado ['mm/s'])
      lecturas: 324,329
[OK] unidad de weight: ['kg'] (esperado ['kg'])
      lecturas: 162,162

Tiempo: 4.1 s


In [14]:
# Hito 2.4: Duplicados por clave y reparto de equipos por worker
t0 = time.perf_counter()

clave = ["sensor_id", "metric", "timestamp"]
duplicados = (
    df_raw.groupBy(clave).count()
    .filter(F.col("count") > 1)
    .agg(F.count("*").alias("claves"), F.sum(F.col("count") - 1).alias("filas_sobrantes"))
    .first()
)

workers_por_equipo = (
    df_raw.groupBy("sensor_id")
    .agg(F.countDistinct("worker_id").alias("n"))
    .filter(F.col("n") > 1)
    .count()
)

por_worker = df_raw.groupBy("worker_id").count().orderBy("worker_id").collect()

tiempos_h2["2.4 duplicados y reparto"] = time.perf_counter() - t0

registrar("claves duplicadas (sensor_id, metric, timestamp)", duplicados["claves"], 0)
registrar("filas sobrantes por duplicado", duplicados["filas_sobrantes"] or 0, 0)
registrar("equipos simulados por mas de un worker", workers_por_equipo, 0)
registrar("workers distintos", len(por_worker), len(archivos))

conteos = [r["count"] for r in por_worker]
print(f"\nEventos por worker: min {min(conteos):,} / max {max(conteos):,} "
      f"(max/min = {max(conteos) / min(conteos):.2f})")
print("El desbalance es esperado: los equipos tienen 1, 2 o 3 puntos de medicion.")
print(f"Tiempo: {tiempos_h2['2.4 duplicados y reparto']:.1f} s")

[OK] claves duplicadas (sensor_id, metric, timestamp): 0 (esperado 0)
[OK] filas sobrantes por duplicado: 0 (esperado 0)
[OK] equipos simulados por mas de un worker: 0 (esperado 0)
[OK] workers distintos: 16 (esperado 16)

Eventos por worker: min 540,540 / max 702,704 (max/min = 1.30)
El desbalance es esperado: los equipos tienen 1, 2 o 3 puntos de medicion.
Tiempo: 6.9 s


In [15]:
# Hito 2.5: Continuidad de las series (huecos a 15 s)
t0 = time.perf_counter()

ticks_esperados = -(-EVENTOS_ESPERADOS // PUNTOS_ESPERADOS)   # division con techo

ts = F.try_to_timestamp("timestamp", F.lit("yyyy-MM-dd'T'HH:mm:ss.SSSX"))   # ansi activo: to_timestamp abortaria el job
series = (
    df_raw.groupBy("sensor_id", "metric")
    .agg(
        F.count("*").alias("lecturas"),
        F.sum(ts.isNull().cast("int")).alias("ts_invalidos"),
        F.min(ts).alias("desde"),
        F.max(ts).alias("hasta"),
    )
    .withColumn(
        "huecos",
        ((F.unix_timestamp("hasta") - F.unix_timestamp("desde")) / DT_S + 1 - F.col("lecturas"))
        .cast("long"),
    )
    .cache()
)

resumen_series = series.agg(
    F.min("lecturas").alias("min_lecturas"),
    F.max("lecturas").alias("max_lecturas"),
    F.sum("huecos").alias("huecos"),
    F.sum((F.col("huecos") < 0).cast("int")).alias("series_con_exceso"),
    F.sum("ts_invalidos").alias("ts_invalidos"),
    F.min("desde").alias("desde"),
    F.max("hasta").alias("hasta"),
).first()

tiempos_h2["2.5 continuidad"] = time.perf_counter() - t0

registrar("lecturas minimas por serie", resumen_series["min_lecturas"], ticks_esperados - 1,
          ok=resumen_series["min_lecturas"] >= ticks_esperados - 1)
registrar("lecturas maximas por serie", resumen_series["max_lecturas"], ticks_esperados)
registrar("huecos en las series", resumen_series["huecos"], 0)
registrar("series con mas lecturas que ticks", resumen_series["series_con_exceso"], 0)
registrar("timestamp con fecha imposible", resumen_series["ts_invalidos"], 0)

dias = (resumen_series["hasta"] - resumen_series["desde"]).total_seconds() / 86400
print(f"\nPeriodo simulado: {resumen_series['desde']} -> {resumen_series['hasta']} ({dias:.2f} dias)")
print(f"Ticks esperados : {ticks_esperados:,} de {DT_S} s")
print(f"Tiempo: {tiempos_h2['2.5 continuidad']:.1f} s")

[Stage 48:=====================================================>(198 + 2) / 200]

[OK] lecturas minimas por serie: 54054 (esperado 54054)
[OK] lecturas maximas por serie: 54055 (esperado 54055)
[OK] huecos en las series: 0 (esperado 0)
[OK] series con mas lecturas que ticks: 0 (esperado 0)
[OK] timestamp con fecha imposible: 0 (esperado 0)

Periodo simulado: 2026-01-01 00:00:00 -> 2026-01-10 09:13:30 (9.38 dias)
Ticks esperados : 54,055 de 15 s
Tiempo: 5.8 s


In [16]:
# Hito 2.5: Series que alcanzan a emitir en el ultimo tick (cortado)
series.filter(F.col("lecturas") == ticks_esperados) \
      .select("sensor_id", "metric", "lecturas", "huecos") \
      .orderBy("sensor_id", "metric").show(truncate=False)
series.unpersist()

+----------+---------+--------+------+
|sensor_id |metric   |lecturas|huecos|
+----------+---------+--------+------+
|FAJA_S3_C1|current  |54055   |0     |
|FAJA_S3_C1|vibration|54055   |0     |
|FAJA_S3_C2|current  |54055   |0     |
|FAJA_S3_C2|vibration|54055   |0     |
|FAJA_S3_C3|current  |54055   |0     |
|FAJA_S3_C3|vibration|54055   |0     |
|FAJA_S3_C4|current  |54055   |0     |
|FAJA_S3_C4|vibration|54055   |0     |
|FAJA_S3_C5|current  |54055   |0     |
|FAJA_S3_C5|vibration|54055   |0     |
+----------+---------+--------+------+



DataFrame[sensor_id: string, metric: string, lecturas: bigint, ts_invalidos: bigint, desde: timestamp, hasta: timestamp, huecos: bigint]

In [17]:
# Hito 2.6: Perfil estadistico de value por metrica
t0 = time.perf_counter()

perfil = (
    df_raw.groupBy("metric")
    .agg(
        F.count("value").alias("n"),
        F.round(F.mean("value"), 3).alias("media"),
        F.round(F.stddev("value"), 3).alias("desv"),
        F.min("value").alias("min"),
        F.percentile_approx("value", [0.01, 0.5, 0.99]).alias("p01_p50_p99"),
        F.max("value").alias("max"),
    )
    .orderBy("metric")
)
perfil.show(truncate=False)

tiempos_h2["2.6 perfil de value"] = time.perf_counter() - t0
print(f"Tiempo: {tiempos_h2['2.6 perfil de value']:.1f} s")

[Stage 58:>                                                         (0 + 1) / 1]

+-----------+-------+---------+---------+------+------------------------+--------+
|metric     |n      |media    |desv     |min   |p01_p50_p99             |max     |
+-----------+-------+---------+---------+------+------------------------+--------+
|current    |3027029|19.611   |46.67    |0.0   |[7.24, 7.52, 282.46]    |403.45  |
|humidity   |3243240|75.521   |3.579    |62.0  |[65.7, 75.0, 89.1]      |93.9    |
|temperature|3243240|-7.038   |13.903   |-18.37|[-18.11, -17.95, 19.28] |20.62   |
|vibration  |324329 |3.402    |1.554    |0.0   |[0.0, 3.481, 6.241]     |6.637   |
|weight     |162162 |13346.972|11807.963|0.0   |[0.0, 13468.1, 40272.46]|41997.91|
+-----------+-------+---------+---------+------+------------------------+--------+

Tiempo: 3.1 s


In [18]:
# Hito 2.6: Temperatura por familia (reefers vs ambientales)
(
    df_raw.filter(F.col("metric") == "temperature")
    .withColumn("familia", F.split("sensor_id", "_").getItem(0))
    .groupBy("familia")
    .agg(
        F.count("*").alias("n"),
        F.min("value").alias("min"),
        F.percentile_approx("value", [0.01, 0.5, 0.99]).alias("p01_p50_p99"),
        F.max("value").alias("max"),
    )
    .orderBy("familia")
    .show(truncate=False)
)

[Stage 59:======================================>                 (11 + 4) / 16]

+-------+-------+------+---------------------+-----+
|familia|n      |min   |p01_p50_p99          |max  |
+-------+-------+------+---------------------+-----+
|AMB    |648648 |9.42  |[9.94, 14.66, 20.1]  |20.62|
|REEFER |2594592|-18.37|[-18.11, -17.98, 4.1]|5.89 |
+-------+-------+------+---------------------+-----+



In [20]:
# Hito 2.7: Resumen del diagnostico y tiempos
print(f"{'Chequeo':<52} {'Medido':>14} {'Esperado':>14}  Estado")
print("-" * 92)
for chequeo, (medido, esperado, ok) in diagnostico.items():
    print(f"{chequeo:<52} {str(medido):>14.14} {str(esperado):>14.14}  {'OK' if ok else 'FALLA'}")

fallas = [c for c, (_, _, ok) in diagnostico.items() if not ok]
print(f"\nChequeos: {len(diagnostico)}  |  OK: {len(diagnostico) - len(fallas)}  |  FALLA: {len(fallas)}")

print("\nTiempos del hito 2:")
for seccion, s in tiempos_h2.items():
    print(f"  {seccion:<28} {s:6.1f} s")
print(f"  {'total':<28} {sum(tiempos_h2.values()):6.1f} s")

Chequeo                                                      Medido       Esperado  Estado
--------------------------------------------------------------------------------------------
lineas corruptas                                                  0              0  OK
nulos en timestamp                                                0              0  OK
nulos en sensor_id                                                0              0  OK
nulos en metric                                                   0              0  OK
nulos en value                                                    0              0  OK
nulos en unit                                                     0              0  OK
nulos en worker_id                                                0              0  OK
timestamp mal formado                                             0              0  OK
value NaN                                                         0              0  OK
value fisicamente imposible      

In [ ]:
# Hito 3: Limpieza, normalizacion y filtrado guiados por el diagnostico del hito 2
from functools import reduce

# series por familia de equipo segun src/simulador.py (_inventario)
SERIES_POR_FAMILIA = {"FAJA": 12, "GRUA": 4, "BASCULA": 1, "REEFER": 144, "AMB": 24}

tiempos_h3 = {}

In [ ]:
# Hito 3.1: Motivo de descarte por linea, el primero que aplique, en una pasada sobre el cache
t0 = time.perf_counter()

# ts (try_to_timestamp) y fuera_de_rango vienen de los hitos 2.5 y 2.2
alguno_nulo = reduce(lambda a, b: a | b, [F.col(c).isNull() for c in columnas])
df_marcado = df_raw.withColumn(
    "motivo_descarte",
    F.when(F.col("_corrupt_record").isNotNull(), "corrupta")
    .when(alguno_nulo, "nulo")
    .when(ts.isNull(), "timestamp invalido")
    .when(F.isnan("value"), "value NaN")
    .when(fuera_de_rango, "fuera de rango fisico"),
)

descartes = {
    r["motivo_descarte"]: r["count"]
    for r in df_marcado.filter(F.col("motivo_descarte").isNotNull())
                       .groupBy("motivo_descarte").count().collect()
}

tiempos_h3["3.1 motivos de descarte"] = time.perf_counter() - t0

print(f"Lineas a descartar: {sum(descartes.values()):,}")
for motivo, n in sorted(descartes.items()):
    print(f"  {motivo:<24} {n:,}")
print(f"Tiempo: {tiempos_h3['3.1 motivos de descarte']:.1f} s")

In [ ]:
# Hito 3.2: DataFrame limpio con timestamp tipado y familia de equipo, sin columnas de diagnostico
df_limpio = df_marcado.filter(F.col("motivo_descarte").isNull()).select(
    ts.alias("timestamp"),
    "sensor_id",
    F.split("sensor_id", "_").getItem(0).alias("asset_type"),
    "metric",
    "value",
    "unit",
    "worker_id",
)

# dropDuplicates cuesta un shuffle de 10M filas: solo si el hito 2.4 encontro duplicados
hay_duplicados = diagnostico["claves duplicadas (sensor_id, metric, timestamp)"][0] > 0
if hay_duplicados:
    df_limpio = df_limpio.dropDuplicates(["sensor_id", "metric", "timestamp"])

print(f"Deduplicacion: {'aplicada' if hay_duplicados else 'omitida, 0 duplicados en el hito 2.4'}")
df_limpio.printSchema()

In [ ]:
# Hito 3.3: Cache del DataFrame limpio, conteo y liberacion de df_raw
t0 = time.perf_counter()
df_limpio = df_limpio.cache()
total_limpio = df_limpio.count()
tiempos_h3["3.3 cache y conteo"] = time.perf_counter() - t0

df_raw.unpersist()   # desde aqui los hitos trabajan sobre df_limpio

sobrantes = diagnostico["filas sobrantes por duplicado"][0] if hay_duplicados else 0
esperado_limpio = total - sum(descartes.values()) - sobrantes

print(f"Registros crudos   : {total:,}")
print(f"Descartados        : {sum(descartes.values()):,}")
print(f"Duplicados quitados: {sobrantes:,}")
print(f"Registros limpios  : {total_limpio:,} (esperado {esperado_limpio:,}) "
      f"[{'OK' if total_limpio == esperado_limpio else 'FALLA'}]")
print(f"Tiempo: {tiempos_h3['3.3 cache y conteo']:.1f} s")

In [ ]:
# Hito 3.4: Verificacion del resultado: tipo y rango del timestamp, series por familia
t0 = time.perf_counter()

rango = df_limpio.agg(
    F.min("timestamp").alias("desde"),
    F.max("timestamp").alias("hasta"),
    F.sum(F.col("timestamp").isNull().cast("int")).alias("ts_nulos"),
).first()

familias = {
    r["asset_type"]: r
    for r in df_limpio.groupBy("asset_type")
                      .agg(F.countDistinct("sensor_id", "metric").alias("series"),
                           F.count("*").alias("lecturas"))
                      .collect()
}

tiempos_h3["3.4 verificacion"] = time.perf_counter() - t0

print(f"Tipo de timestamp: {df_limpio.schema['timestamp'].dataType.simpleString()}")
print(f"timestamp nulos  : {rango['ts_nulos']}")
print(f"Periodo          : {rango['desde']} -> {rango['hasta']}")
print(f"Periodo hito 2.5 : {resumen_series['desde']} -> {resumen_series['hasta']}")
print()
for familia, esperadas in SERIES_POR_FAMILIA.items():
    series_f = familias[familia]["series"] if familia in familias else 0
    lecturas_f = familias[familia]["lecturas"] if familia in familias else 0
    print(f"[{'OK' if series_f == esperadas else 'FALLA'}] {familia:<8} series {series_f:>3} "
          f"(esperadas {esperadas:>3})  lecturas {lecturas_f:>10,}")
otras = set(familias) - set(SERIES_POR_FAMILIA)
if otras:
    print(f"[FALLA] familias no esperadas: {sorted(otras)}")
print(f"Tiempo: {tiempos_h3['3.4 verificacion']:.1f} s")

In [ ]:
# Hito 3.5: Muestra del DataFrame limpio y tiempos del hito 3
df_limpio.show(5, truncate=False)

print("Tiempos del hito 3:")
for seccion, s in tiempos_h3.items():
    print(f"  {seccion:<28} {s:6.1f} s")
print(f"  {'total':<28} {sum(tiempos_h3.values()):6.1f} s")

In [ ]:
# Hito 4: Diagnostico y optimizacion del shuffle
import json
import statistics
from urllib.request import urlopen

URL_API = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}"   # api rest de la ui del driver
REPETICIONES_H4 = 3
tiempos_h4 = {}


def etapas_del_grupo(grupo):
    # etapas completadas de los jobs lanzados con setJobGroup(grupo)
    jobs = [j for j in json.load(urlopen(f"{URL_API}/jobs")) if j.get("jobGroup") == grupo]
    ids = {s for j in jobs for s in j["stageIds"]}
    return sorted(
        (s for s in json.load(urlopen(f"{URL_API}/stages")) if s["stageId"] in ids and s["status"] == "COMPLETE"),
        key=lambda s: s["stageId"],
    )

In [ ]:
# Hito 4.1: Balance de las particiones de entrada de df_limpio
t0 = time.perf_counter()
filas_por_particion = [r["count"] for r in df_limpio.groupBy(F.spark_partition_id().alias("p")).count().collect()]
tiempos_h4["4.1 balance de entrada"] = time.perf_counter() - t0

print(f"Particiones de entrada: {len(filas_por_particion)}")
print(f"Filas por particion   : min {min(filas_por_particion):,} / max {max(filas_por_particion):,} "
      f"(max/min = {max(filas_por_particion) / min(filas_por_particion):.2f})")

In [ ]:
# Hito 4.2: Consulta de referencia (resumen horario por serie) y su plan con el Exchange
def resumen_horario(df):
    return (
        df.groupBy("sensor_id", "metric", F.date_trunc("hour", "timestamp").alias("hora"))
        .agg(F.count("*").alias("n"), F.avg("value").alias("media"),
             F.min("value").alias("min"), F.max("value").alias("max"))
    )


spark.conf.set("spark.sql.adaptive.enabled", "false")   # plan sin AQE para ver el Exchange tal cual
resumen_horario(df_limpio).explain()
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [ ]:
# Hito 4.3: Misma consulta con 4 configuraciones de shuffle, mediana de 3 corridas (accion noop)
CONFIGS_H4 = [
    ("200 sin AQE", 200, "false"),
    ("200 con AQE", 200, "true"),
    (f"{sc.defaultParallelism} sin AQE", sc.defaultParallelism, "false"),
    (f"{2 * sc.defaultParallelism} sin AQE", 2 * sc.defaultParallelism, "false"),
]

t0_h4 = time.perf_counter()
resultados_h4 = []
for nombre, particiones, aqe in CONFIGS_H4:
    spark.conf.set("spark.sql.shuffle.partitions", particiones)
    spark.conf.set("spark.sql.adaptive.enabled", aqe)
    corridas = []
    for i in range(REPETICIONES_H4):
        grupo = f"h4-{nombre}-{i}"
        sc.setJobGroup(grupo, nombre)
        t0 = time.perf_counter()
        resumen_horario(df_limpio).write.format("noop").mode("overwrite").save()
        corridas.append(time.perf_counter() - t0)
    etapas = etapas_del_grupo(grupo)   # metricas de la ultima corrida
    resultados_h4.append({
        "config": nombre,
        "particiones": particiones,
        "aqe": aqe,
        "mediana_s": statistics.median(corridas),
        "tareas_por_etapa": [s["numTasks"] for s in etapas],
        "shuffle_mib": sum(s["shuffleWriteBytes"] for s in etapas) / 1024**2,
    })
sc.setLocalProperty("spark.jobGroup.id", None)   # fin del agrupamiento de jobs
tiempos_h4["4.3 comparacion de configs"] = time.perf_counter() - t0_h4

print(f"{'Config':<14} {'Mediana':>8} {'Shuffle':>10}  Tareas por etapa")
print("-" * 60)
for r in resultados_h4:
    print(f"{r['config']:<14} {r['mediana_s']:7.2f}s {r['shuffle_mib']:8.1f}MiB  {r['tareas_por_etapa']}")

In [ ]:
# Hito 4.4: Se aplica la configuracion mas rapida para el resto del notebook
ganadora = min(resultados_h4, key=lambda r: r["mediana_s"])
base = next(r for r in resultados_h4 if r["config"] == "200 sin AQE")

spark.conf.set("spark.sql.shuffle.partitions", ganadora["particiones"])
spark.conf.set("spark.sql.adaptive.enabled", ganadora["aqe"])

print(f"Configuracion elegida: {ganadora['config']} ({ganadora['mediana_s']:.2f} s)")
print(f"Mejora vs 200 sin AQE: {base['mediana_s'] / ganadora['mediana_s']:.2f}x")
print(f"shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')}, "
      f"AQE = {spark.conf.get('spark.sql.adaptive.enabled')}")

print("\nTiempos del hito 4:")
for seccion, s in tiempos_h4.items():
    print(f"  {seccion:<28} {s:6.1f} s")
print(f"  {'total':<28} {sum(tiempos_h4.values()):6.1f} s")

In [ ]:
# Hito 5: Escritura en Parquet particionado por fecha
RUTA_PARQUET = f"{RUTA_SALIDA}/eventos_limpios"
tiempos_h5 = {}

In [ ]:
# Hito 5.1: Escritura: un archivo por fecha, ordenado por serie para comprimir mejor
t0 = time.perf_counter()
(
    df_limpio.withColumn("fecha", F.to_date("timestamp"))
    .repartition("fecha")
    .sortWithinPartitions("sensor_id", "metric", "timestamp")
    .write.mode("overwrite")
    .partitionBy("fecha")
    .parquet(RUTA_PARQUET)
)
tiempos_h5["5.1 escritura"] = time.perf_counter() - t0
print(f"Escrito en {RUTA_PARQUET} en {tiempos_h5['5.1 escritura']:.1f} s")

In [ ]:
# Hito 5.2: Inventario en disco y compresion frente al JSONL de entrada
parquets = sorted(glob.glob(f"{RUTA_PARQUET}/fecha=*/*.parquet"))
fechas = sorted({os.path.basename(os.path.dirname(p)) for p in parquets})
bytes_parquet = sum(os.path.getsize(p) for p in parquets)

print(f"Particiones fecha : {len(fechas)} ({fechas[0]} .. {fechas[-1]})")
print(f"Archivos parquet  : {len(parquets)}")
print(f"Volumen parquet   : {bytes_parquet / 1024**2:,.1f} MiB")
print(f"Volumen JSONL     : {total_bytes / 1024**2:,.1f} MiB")
print(f"Reduccion         : {total_bytes / bytes_parquet:.1f}x")
print(f"_SUCCESS presente : {os.path.exists(f'{RUTA_PARQUET}/_SUCCESS')}")

In [ ]:
# Hito 5.3: Relectura y verificacion: conteo, esquema y suma de control contra df_limpio
t0 = time.perf_counter()
df_parquet = spark.read.parquet(RUTA_PARQUET)
total_parquet = df_parquet.count()
tiempos_h5["5.3 relectura (count)"] = time.perf_counter() - t0

def control(df):
    return df.agg(F.sum("value").alias("suma_value"), F.min("timestamp").alias("desde"),
                  F.max("timestamp").alias("hasta"), F.countDistinct("sensor_id", "metric").alias("series")).first()

t0 = time.perf_counter()
control_limpio, control_parquet = control(df_limpio), control(df_parquet)
tiempos_h5["5.3 suma de control"] = time.perf_counter() - t0

columnas_ok = [c for c in df_parquet.columns if c != "fecha"] == df_limpio.columns
print(f"[{'OK' if total_parquet == total_limpio else 'FALLA'}] registros: {total_parquet:,} (esperado {total_limpio:,})")
print(f"[{'OK' if columnas_ok else 'FALLA'}] columnas: {df_parquet.columns}")
# la suma en coma flotante depende del orden: se compara con tolerancia relativa
control_ok = (
    abs(control_parquet["suma_value"] - control_limpio["suma_value"]) <= 1e-9 * abs(control_limpio["suma_value"])
    and [control_parquet[k] for k in ("desde", "hasta", "series")] == [control_limpio[k] for k in ("desde", "hasta", "series")]
)
print(f"[{'OK' if control_ok else 'FALLA'}] control: {control_parquet.asDict()}")
print(f"Tiempo count parquet: {tiempos_h5['5.3 relectura (count)']:.1f} s")

In [ ]:
# Hito 5.4: Poda de particiones: un dia lee solo su carpeta
dia = fechas[len(fechas) // 2].split("=")[1]
consulta_dia = df_parquet.filter(F.col("fecha") == dia)

t0 = time.perf_counter()
filas_dia = consulta_dia.count()
tiempos_h5["5.4 consulta de un dia"] = time.perf_counter() - t0

plan = consulta_dia._jdf.queryExecution().executedPlan().toString()
poda = [l.strip() for l in plan.split("\n") if "PartitionFilters" in l]
print(f"Dia {dia}: {filas_dia:,} filas en {tiempos_h5['5.4 consulta de un dia']:.2f} s")
print(f"[{'OK' if poda and 'fecha' in poda[0] else 'FALLA'}] PartitionFilters en el plan")
print(poda[0][poda[0].find("PartitionFilters"):][:120] if poda else "sin PartitionFilters")

In [ ]:
# Hito 5.5: Tiempos del hito 5
print("Tiempos del hito 5:")
for seccion, s in tiempos_h5.items():
    print(f"  {seccion:<28} {s:6.1f} s")
print(f"  {'total':<28} {sum(tiempos_h5.values()):6.1f} s")